# Does the pleiotropy ceiling replicate on Pharmaprojects?

Phase 2 tested the 2–5 therapeutic-area window as one block. But the window is two claims, and only
the second is interesting:

- the **floor** (TA ≥ 2) removes targets seen in a single therapeutic area, which are mostly targets
  with very little genetic evidence at all;
- the **ceiling** (TA ≤ 5) removes *highly pleiotropic* targets. That is the substantive claim — broad
  pleiotropy predicts failure — and it is what the window is doing.

So the question is not whether "2–5" transfers but whether **excluding high-pleiotropy targets raises
enrichment in Pharmaprojects too**. This notebook tests the ceiling on its own, in both resources,
with the same specification.

Inputs are the master tables from `01_build_pair_tables.ipynb`; statistics from `or10_stats.py`.

In [1]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

from or10_stats import or_rs, support_mask

pd.set_option("display.width", 240)
pd.set_option("display.max_columns", 40)

path_to_intermediate_data_folder = "../../../data/intermediate_files/"
chembl = pd.read_parquet(path_to_intermediate_data_folder + "ti_pairs_chembl_master-r1.parquet")
pp = pd.read_parquet(path_to_intermediate_data_folder + "ti_pairs_pharmaprojects_master-r1.parquet")

for df in (chembl, pp):
    df["ta"] = df["uniqueTherapeuticAreas"].fillna(0).astype(float)
    df["gps"] = df["uniqueDiseases"].fillna(0).astype(float)
    df["support_all"] = support_mask(df).astype(int)
    df["support_pav"] = support_mask(df, pav=True).astype(int)

DATASETS = [("ChEMBL", chembl), ("Pharmaprojects", pp)]
for label, df in DATASETS:
    print(
        f"{label}: {len(df)} pairs, {int(df['approved'].sum())} approved/launched, "
        f"{int(df['support_all'].sum())} with support ({int(df['support_pav'].sum())} PAV)"
    )

ChEMBL: 37377 pairs, 4564 approved/launched, 742 with support (161 PAV)
Pharmaprojects: 7390 pairs, 913 approved/launched, 469 with support (137 PAV)


## The comparison

Three groups per support type, and the no-support pairs are the common reference so the two supported
groups are not each other's control:

| group | definition |
| ----- | ---------- |
| `no support` | reference |
| `low pleiotropy` | supported, 2 ≤ TA ≤ 5 |
| `high pleiotropy` | supported, TA ≥ 6 |

Fisher gives each group's odds ratio against no support; a logistic regression on the three-level
factor gives the P value for the difference between the two supported groups, by a t-test on the
contrast — the specification the published stratified comparisons used.

Run twice per resource: on PAV-supported pairs (the published definition's support type) and on all
supported pairs (weaker signal, more pairs, so more power for the ceiling itself). Pairs with
supported TA ≤ 1 are excluded from this comparison and counted separately, so the contrast is purely
the ceiling and not partly the floor.

In [2]:
def ceiling_test(df, dataset, support_col, support_label, split_at=6, low_min=2):
    """Low versus high pleiotropy among supported pairs, against the no-support reference."""
    supported = df[support_col] == 1
    none = df[support_col] == 0
    low = supported & df["in_gps"] & (df["ta"] >= low_min) & (df["ta"] < split_at)
    high = supported & df["in_gps"] & (df["ta"] >= split_at)
    dropped = supported & ~(low | high)

    rows = []
    for group, mask in [("low pleiotropy", low), ("high pleiotropy", high)]:
        sub = df[mask | none]
        res = or_rs(mask.loc[sub.index], sub["approved"])
        rows.append(
            {
                "dataset": dataset,
                "support": support_label,
                "group": group,
                "odds_ratio": res["odds_ratio"],
                "ci_low": res["ci_low"],
                "ci_high": res["ci_high"],
                "relative_success": res["relative_success"],
                "n_pairs": res["n_support"],
                "n_approved": res["yes_evid-high_clinphase"],
            }
        )

    e = np.where(low, 2, np.where(high, 1, 0))
    data = pd.DataFrame({"outcome": df["approved"].to_numpy(), "E": e})[~dropped.to_numpy()]
    fit = smf.logit("outcome ~ C(E)", data=data).fit(disp=False)
    contrast = np.zeros(len(fit.params))
    contrast[1], contrast[2] = -1, 1  # low minus high
    contrast_p = float(np.ravel(fit.t_test(contrast).pvalue)[0])

    summary = {
        "dataset": dataset,
        "support": support_label,
        "or_low": float(np.exp(fit.params.iloc[2])),
        "or_high": float(np.exp(fit.params.iloc[1])),
        "ratio_low_over_high": float(np.exp(fit.params.iloc[2] - fit.params.iloc[1])),
        "p_difference": contrast_p,
        "n_low": int((e == 2).sum()),
        "n_high": int((e == 1).sum()),
        "n_approved_low": int(data.loc[data["E"] == 2, "outcome"].sum()),
        "n_approved_high": int(data.loc[data["E"] == 1, "outcome"].sum()),
        "n_supported_dropped_ta_le_1": int(dropped.sum()),
    }
    return pd.DataFrame(rows), summary


strata, summaries = [], []
for label, df in DATASETS:
    for support_col, support_label in [("support_pav", "PAV support"), ("support_all", "any support")]:
        rows, summary = ceiling_test(df, label, support_col, support_label)
        strata.append(rows)
        summaries.append(summary)

strata = pd.concat(strata, ignore_index=True)
ceiling = pd.DataFrame(summaries)
strata.round(3)

,dataset,support,group,odds_ratio,ci_low,ci_high,relative_success,n_pairs,n_approved
0,ChEMBL,PAV support,low pleiotropy,10.320,6.728,15.830,4.857,87,51
1,ChEMBL,PAV support,high pleiotropy,3.100,1.835,5.236,2.473,67,20
2,ChEMBL,any support,low pleiotropy,4.012,3.257,4.943,2.960,398,139
3,ChEMBL,any support,high pleiotropy,2.969,2.290,3.848,2.409,285,81
4,Pharmaprojects,PAV support,low pleiotropy,4.502,2.663,7.612,3.159,60,23
5,Pharmaprojects,PAV support,high pleiotropy,1.086,0.537,2.197,1.075,69,9
6,Pharmaprojects,any support,low pleiotropy,1.877,1.322,2.664,1.699,202,41
7,Pharmaprojects,any support,high pleiotropy,1.481,1.042,2.105,1.401,233,39


In [3]:
print("low (TA 2-5) versus high (TA >= 6) pleiotropy among supported pairs:")
print(
    ceiling[
        [
            "dataset",
            "support",
            "or_low",
            "or_high",
            "ratio_low_over_high",
            "p_difference",
            "n_low",
            "n_high",
            "n_approved_low",
            "n_approved_high",
        ]
    ]
    .round(4)
    .to_string(index=False)
)
print()
for _, r in ceiling.iterrows():
    verdict = (
        "replicates"
        if (r["ratio_low_over_high"] > 1 and r["p_difference"] < 0.05)
        else ("same direction, not significant" if r["ratio_low_over_high"] > 1 else "opposite direction")
    )
    print(
        f"{r['dataset']:15s} {r['support']:12s}: low/high = {r['ratio_low_over_high']:.2f}, "
        f"p = {r['p_difference']:.3g}  -> {verdict}"
    )

low (TA 2-5) versus high (TA >= 6) pleiotropy among supported pairs:
       dataset     support  or_low  or_high  ratio_low_over_high  p_difference  n_low  n_high  n_approved_low  n_approved_high
        ChEMBL PAV support 10.3203   3.1000               3.3292        0.0005     87      67              51               20
        ChEMBL any support  4.0124   2.9686               1.3516        0.0733    398     285             139               81
Pharmaprojects PAV support  4.5018   1.0863               4.1441        0.0014     60      69              23                9
Pharmaprojects any support  1.8765   1.4814               1.2668        0.3399    202     233              41               39

ChEMBL          PAV support : low/high = 3.33, p = 0.00048  -> replicates
ChEMBL          any support : low/high = 1.35, p = 0.0733  -> same direction, not significant
Pharmaprojects  PAV support : low/high = 4.14, p = 0.00141  -> replicates
Pharmaprojects  any support : low/high = 1.27, p = 0.

## Where the ceiling sits

The 6 split is inherited from the published window, so it is itself a threshold. This sweeps it: for
every cut point, supported pairs below it versus supported pairs at or above it. If the ceiling is
real in a resource, the ratio should exceed 1 across a range of cut points, not at one.

In [4]:
sweep_rows = []
for label, df in DATASETS:
    for support_col, support_label in [("support_pav", "PAV support"), ("support_all", "any support")]:
        for cut in range(3, 11):
            _, summary = ceiling_test(df, label, support_col, support_label, split_at=cut, low_min=2)
            if min(summary["n_approved_low"], summary["n_approved_high"]) == 0:
                continue
            sweep_rows.append({"cut": cut, **summary})
sweep = pd.DataFrame(sweep_rows)

for (label, support_label), sub in sweep.groupby(["dataset", "support"], sort=False):
    print(f"{label} / {support_label}")
    print(
        sub[["cut", "or_low", "or_high", "ratio_low_over_high", "p_difference", "n_approved_low", "n_approved_high"]]
        .round(3)
        .to_string(index=False)
    )
    print()

ChEMBL / PAV support
 cut  or_low  or_high  ratio_low_over_high  p_difference  n_approved_low  n_approved_high
   3  18.212    5.625                3.238         0.056              10               61
   4  10.927    5.438                2.009         0.092              18               53
   5  10.407    4.106                2.535         0.005              40               31
   6  10.320    3.100                3.329         0.000              51               20
   7   9.487    2.732                3.473         0.001              56               15
   8   8.406    2.585                3.252         0.003              60               11
   9   7.924    2.522                3.142         0.007              62                9
  10   7.524    2.649                2.840         0.020              63                8

ChEMBL / any support
 cut  or_low  or_high  ratio_low_over_high  p_difference  n_approved_low  n_approved_high
   3   4.846    3.382                1.433         0.125 

## Enrichment by pleiotropy bin

The same thing without any dichotomy: odds ratio of each pleiotropy bin against the no-support
reference. This is the shape the window was built on, and the question is whether Pharmaprojects shows
the same decline at the top end. Counts are printed against every bin because the PAV-supported bins
in Pharmaprojects are thin.

In [5]:
BINS = [(1, 1, "TA 1"), (2, 3, "TA 2-3"), (4, 5, "TA 4-5"), (6, 9, "TA 6-9"), (10, None, "TA 10+")]

bin_rows = []
for label, df in DATASETS:
    for support_col, support_label in [("support_pav", "PAV support"), ("support_all", "any support")]:
        none = df[support_col] == 0
        for lo, hi, name in BINS:
            mask = (df[support_col] == 1) & df["in_gps"] & (df["ta"] >= lo)
            if hi is not None:
                mask = mask & (df["ta"] <= hi)
            sub = df[mask | none]
            res = or_rs(mask.loc[sub.index], sub["approved"])
            bin_rows.append(
                {
                    "dataset": label,
                    "support": support_label,
                    "bin": name,
                    "odds_ratio": res["odds_ratio"],
                    "ci_low": res["ci_low"],
                    "ci_high": res["ci_high"],
                    "relative_success": res["relative_success"],
                    "n_pairs": res["n_support"],
                    "n_approved": res["yes_evid-high_clinphase"],
                }
            )
bins = pd.DataFrame(bin_rows)

for (label, support_label), sub in bins.groupby(["dataset", "support"], sort=False):
    print(f"{label} / {support_label}")
    print(
        sub[["bin", "odds_ratio", "ci_low", "ci_high", "relative_success", "n_pairs", "n_approved"]]
        .round(3)
        .to_string(index=False)
    )
    print()

ChEMBL / PAV support
   bin  odds_ratio  ci_low  ci_high  relative_success  n_pairs  n_approved
  TA 1       1.214   0.146   10.088             1.184        7           1
TA 2-3      10.927   5.260   22.701             4.971       30          18
TA 4-5      10.017   5.915   16.962             4.797       57          33
TA 6-9       3.497   1.756    6.965             2.687       37          12
TA 10+       2.649   1.179    5.954             2.209       30           8

ChEMBL / any support
   bin  odds_ratio  ci_low  ci_high  relative_success  n_pairs  n_approved
  TA 1       4.445   2.620    7.542             3.161       59          22
TA 2-3       4.332   3.241    5.789             3.109      199          73
TA 4-5       3.710   2.757    4.992             2.811      199          66
TA 6-9       3.151   2.268    4.378             2.513      172          51
TA 10+       2.702   1.778    4.108             2.250      113          30

Pharmaprojects / PAV support
   bin  odds_ratio  ci_low 

## The same question in gPS

The manuscript also states the pleiotropy effect in gPS (distinct diseases): gPS ≤ 5 gives OR 4.8
versus 3.0 for gPS ≥ 10, P = 0.008. Testing that split in both resources, since gPS has a finer grain
than therapeutic areas and therefore more room to show a ceiling.

In [6]:
gps_rows = []
gps_contrasts = []
for label, df in DATASETS:
    for support_col, support_label in [("support_pav", "PAV support"), ("support_all", "any support")]:
        none = df[support_col] == 0
        low = (df[support_col] == 1) & df["in_gps"] & (df["gps"] <= 5)
        high = (df[support_col] == 1) & df["in_gps"] & (df["gps"] >= 10)
        for name, mask in [("gPS <= 5", low), ("gPS >= 10", high)]:
            sub = df[mask | none]
            res = or_rs(mask.loc[sub.index], sub["approved"])
            gps_rows.append(
                {
                    "dataset": label,
                    "support": support_label,
                    "group": name,
                    "odds_ratio": res["odds_ratio"],
                    "ci_low": res["ci_low"],
                    "ci_high": res["ci_high"],
                    "n_pairs": res["n_support"],
                    "n_approved": res["yes_evid-high_clinphase"],
                }
            )
        e = np.where(low, 2, np.where(high, 1, 0))
        keep = ~((df[support_col] == 1) & ~(low | high)).to_numpy()
        data = pd.DataFrame({"outcome": df["approved"].to_numpy(), "E": e})[keep]
        fit = smf.logit("outcome ~ C(E)", data=data).fit(disp=False)
        contrast = np.zeros(len(fit.params))
        contrast[1], contrast[2] = -1, 1
        p_diff = float(np.ravel(fit.t_test(contrast).pvalue)[0])
        gps_contrasts.append(
            {
                "dataset": label,
                "support": support_label,
                "or_low": float(np.exp(fit.params.iloc[2])),
                "or_high": float(np.exp(fit.params.iloc[1])),
                "ratio_low_over_high": float(np.exp(fit.params.iloc[2] - fit.params.iloc[1])),
                "p_difference": p_diff,
                "n_low": int((e[keep] == 2).sum()),
                "n_high": int((e[keep] == 1).sum()),
                "n_approved_low": int(data.loc[data["E"] == 2, "outcome"].sum()),
                "n_approved_high": int(data.loc[data["E"] == 1, "outcome"].sum()),
            }
        )
        print(
            f"{label:15s} {support_label:12s}: gPS<=5 OR {np.exp(fit.params.iloc[2]):.2f} versus "
            f"gPS>=10 OR {np.exp(fit.params.iloc[1]):.2f}, ratio "
            f"{np.exp(fit.params.iloc[2] - fit.params.iloc[1]):.2f}, p = {p_diff:.3g}"
        )

gps_table = pd.DataFrame(gps_rows)
gps_contrast = pd.DataFrame(gps_contrasts)
print()
print(gps_contrast.round(4).to_string(index=False))
print()
print(gps_table.round(3).to_string(index=False))

ChEMBL          PAV support : gPS<=5 OR 9.11 versus gPS>=10 OR 4.69, ratio 1.94, p = 0.0929
ChEMBL          any support : gPS<=5 OR 4.80 versus gPS>=10 OR 2.97, ratio 1.62, p = 0.00772
Pharmaprojects  PAV support : gPS<=5 OR 3.79 versus gPS>=10 OR 1.43, ratio 2.66, p = 0.039
Pharmaprojects  any support : gPS<=5 OR 1.92 versus gPS>=10 OR 1.62, ratio 1.18, p = 0.542

       dataset     support  or_low  or_high  ratio_low_over_high  p_difference  n_low  n_high  n_approved_low  n_approved_high
        ChEMBL PAV support  9.1062   4.6920               1.9408        0.0929     36      97              20               38
        ChEMBL any support  4.7983   2.9677               1.6168        0.0077    220     366              86              104
Pharmaprojects PAV support  3.7935   1.4280               2.6565        0.0390     32      85              11               14
Pharmaprojects any support  1.9190   1.6225               1.1827        0.5423    121     266              25               

## Verdict

In [7]:
for support_label in ["PAV support", "any support"]:
    c = ceiling[(ceiling["dataset"] == "ChEMBL") & (ceiling["support"] == support_label)].iloc[0]
    p = ceiling[(ceiling["dataset"] == "Pharmaprojects") & (ceiling["support"] == support_label)].iloc[0]
    print(f"{support_label}:")
    print(
        f"  ChEMBL         low/high = {c['ratio_low_over_high']:.2f} (p = {c['p_difference']:.3g}), "
        f"OR {c['or_low']:.2f} versus {c['or_high']:.2f}, "
        f"{c['n_approved_low']}/{c['n_low']} versus {c['n_approved_high']}/{c['n_high']} approved"
    )
    print(
        f"  Pharmaprojects low/high = {p['ratio_low_over_high']:.2f} (p = {p['p_difference']:.3g}), "
        f"OR {p['or_low']:.2f} versus {p['or_high']:.2f}, "
        f"{p['n_approved_low']}/{p['n_low']} versus {p['n_approved_high']}/{p['n_high']} launched"
    )
    sweep_sub = sweep[(sweep["dataset"] == "Pharmaprojects") & (sweep["support"] == support_label)]
    print(
        f"  Pharmaprojects cut-point sweep: ratio > 1 at {int((sweep_sub['ratio_low_over_high'] > 1).sum())} "
        f"of {len(sweep_sub)} cut points, p < 0.05 at {int((sweep_sub['p_difference'] < 0.05).sum())}"
    )
    for _, gc in gps_contrast[gps_contrast["support"] == support_label].iterrows():
        print(
            f"  gPS <=5 versus >=10, {gc['dataset']:15s} ratio {gc['ratio_low_over_high']:.2f} "
            f"(OR {gc['or_low']:.2f} versus {gc['or_high']:.2f}), p = {gc['p_difference']:.3g}"
        )
    print()

PAV support:
  ChEMBL         low/high = 3.33 (p = 0.00048), OR 10.32 versus 3.10, 51/87 versus 20/67 approved
  Pharmaprojects low/high = 4.14 (p = 0.00141), OR 4.50 versus 1.09, 23/60 versus 9/69 launched
  Pharmaprojects cut-point sweep: ratio > 1 at 8 of 8 cut points, p < 0.05 at 7
  gPS <=5 versus >=10, ChEMBL          ratio 1.94 (OR 9.11 versus 4.69), p = 0.0929
  gPS <=5 versus >=10, Pharmaprojects  ratio 2.66 (OR 3.79 versus 1.43), p = 0.039

any support:
  ChEMBL         low/high = 1.35 (p = 0.0733), OR 4.01 versus 2.97, 139/398 versus 81/285 approved
  Pharmaprojects low/high = 1.27 (p = 0.34), OR 1.88 versus 1.48, 41/202 versus 39/233 launched
  Pharmaprojects cut-point sweep: ratio > 1 at 7 of 8 cut points, p < 0.05 at 1
  gPS <=5 versus >=10, ChEMBL          ratio 1.62 (OR 4.80 versus 2.97), p = 0.00772
  gPS <=5 versus >=10, Pharmaprojects  ratio 1.18 (OR 1.92 versus 1.62), p = 0.542



## Export

In [8]:
strata.to_csv(path_to_intermediate_data_folder + "or10_ceiling_strata-r1.csv", index=False)
ceiling.to_csv(path_to_intermediate_data_folder + "or10_ceiling_test-r1.csv", index=False)
sweep.to_csv(path_to_intermediate_data_folder + "or10_ceiling_cut_sweep-r1.csv", index=False)
bins.to_csv(path_to_intermediate_data_folder + "or10_ceiling_bins-r1.csv", index=False)
gps_table.to_csv(path_to_intermediate_data_folder + "or10_ceiling_gps-r1.csv", index=False)
gps_contrast.to_csv(path_to_intermediate_data_folder + "or10_ceiling_gps_contrast-r1.csv", index=False)
print("exported 6 tables")

exported 6 tables
